# Лабораторная 3 — Детектор спама (синтетический bag-of-words)

**Тема:** Multinomial Naive Bayes как генеративная модель текста и сравнение с логистической регрессией.  
**Формат данных:** bag-of-words (мешок слов) — вектор счётчиков слов.

## Что такое bag-of-words (мешок слов)
Мы фиксируем словарь из $V$ «слов» (в этой лабораторной это **просто индексы** $1,\dots,V$).  
Документ — это вектор $x\in\mathbb N^V$, где $x_j$ — сколько раз встретилось слово $j$.

Мы **не** строим настоящие тексты и **не** используем `CountVectorizer`: всё синтетически.

## Что нужно сдать
1) Этот ноутбук с реализованными функциями генерации данных для трёх «миров».  
2) Для каждого мира — таблицу метрик (выведет протокол) и короткое объяснение (3–6 предложений).

---

## Важная часть: три «мира»
Вы реализуете три генератора датасетов с одинаковым API:

- `make_world_A(n, V, random_state) -> (X, y)`
- `make_world_B(n, V, random_state) -> (X, y)`
- `make_world_C(n, V, random_state) -> (X, y)`

где:
- $X\in\mathbb N^{n\times V}$ — матрица счётчиков,
- $y\in\{0,1\}^n$ — метки (0 = ham, 1 = spam).

Далее один и тот же протокол сравнивает модели:
- `MultinomialNB(alpha=...)`
- `LogisticRegression` (pipeline со scaling)

---

## Критерий «удалось/не удалось»
В каждом мире **не требуется** абсолютная победа одной модели на всех метриках. Требуется получить **задуманный эффект** и объяснить его.

- Мир A: NB в «правильно специфицированном» мире — обычно хорош по logloss и accuracy.
- Мир B: NB становится мисспецифицированным (например, «два типа спама») — LogReg обычно выигрывает по logloss.
- Мир C: вы делаете ситуацию, где **alpha критически важен** (сильная разреженность + редкие слова + маленькие выборки на фолдах), и показываете, что разные alpha дают заметно разные logloss.

---

## Подсказка: почему Мир C НЕ «два спама»
В мире C **распределение класса одно**, как в мире A.  
Отличие в том, что слова/триггеры настолько редкие и документы настолько короткие (при большом $V$), что в некоторых фолдах обучения
часть слов вообще **не встречается** внутри класса. Тогда без сглаживания ($\alpha\approx 0$) NB получает нулевые вероятности и logloss резко ухудшается.


## Настройка окружения

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB


## Протокол (не менять)

Этот код **унифицирует** сравнение моделей для всех миров.  
Вы должны только вызвать `run_world(make_world_*, "World *")` после того, как реализуете генераторы.


In [61]:
def evaluate_models_clf(X, y, models, *, cv_splits=5, random_state=0):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    scoring = {"acc": "accuracy", "nll": "neg_log_loss"}  # требует predict_proba

    rows = []
    for name, model in models.items():
        out = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
        rows.append({
            "model": name,
            "acc_mean": float(np.mean(out["test_acc"])),
            "acc_std":  float(np.std(out["test_acc"])),
            "logloss_mean": float(np.mean(-out["test_nll"])),
            "logloss_std":  float(np.std(-out["test_nll"])),
        })
    df = pd.DataFrame(rows).sort_values(["logloss_mean", "acc_mean"], ascending=[True, False]).reset_index(drop=True)
    return df

models = {
    "NB alpha=1e-10": MultinomialNB(alpha=1e-10),
    "NB alpha=1":     MultinomialNB(alpha=1.0),
    "NB alpha=10":    MultinomialNB(alpha=10.0),
    "LogReg (scaling)": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(max_iter=3000))
    ]),
}

def sanity_check(X, y):
    X = np.asarray(X)
    y = np.asarray(y)
    assert X.ndim == 2, "X должен быть матрицей n×V"
    assert y.ndim == 1 and len(y) == X.shape[0], "y должен быть длины n"
    # допускаем int dtype или float, если он целочисленный по значениям
    assert np.all(np.equal(X, np.round(X))), "X должен быть целочисленным (счётчики)"
    assert X.min() >= 0, "счётчики не могут быть отрицательными"
    u = np.unique(y)
    assert set(u).issubset({0,1}), f"y должен быть 0/1, сейчас {u}"
    return True

def run_world(make_fn, title, *, n=600, V=800, seed=0, cv_splits=5):
    X, y = make_fn(n=n, V=V, random_state=seed)
    sanity_check(X, y)

    print(f"=== {title} ===")
    print("X shape:", X.shape, "| class balance y.mean():", float(np.mean(y)))
    doc_len = X.sum(axis=1)
    print("doc length: mean", float(doc_len.mean()), "std", float(doc_len.std()),
          "| min", int(doc_len.min()), "max", int(doc_len.max()))
    sparsity = 1.0 - (X != 0).mean()
    print("sparsity (fraction of zeros):", f"{sparsity:.2%}")

    df = evaluate_models_clf(X, y, models, cv_splits=cv_splits, random_state=seed)
    display(df)
    return X, y, df


## Мир A: «Классический спам» (мир, где MultinomialNB оправдан)

### Цель
Сгенерировать данные, которые хорошо соответствуют предпосылке Multinomial Naive Bayes:
$$x\mid y=k\sim \mathrm{Multinomial}(N,\theta_k),\qquad \theta_k \text{ фиксировано для класса } k.$$

### Рецепт генерации (без кода, но строго по шагам)

1) **Сгенерируйте метки классов**
$$y_i\sim \mathrm{Bernoulli}(0.5),\quad i=1,\dots,n.$$

2) **Выберите индексы “спам-триггеров”**  
Выберите множество
$$S_{\text{spam}}\subset\{1,\dots,V\}$$
размера, например, 20–50.  
(Опционально) выберите
$$S_{\text{ham}}$$
для “ham-слов”.

3) **Постройте два распределения слов $\theta_0,\theta_1$**  
- начните с положительных весов $w^{(0)}\in\mathbb R_+^V$, $w^{(1)}\in\mathbb R_+^V$ (например, Gamma или uniform$(0,1)$);  
- увеличьте веса на $S_{\text{spam}}$ для класса spam (например, умножьте на коэффициент $>1$);  
- (опционально) измените веса на $S_{\text{ham}}$ для класса ham;  
- нормируйте:
$$\theta_k=\frac{w^{(k)}}{\sum_{j=1}^V w^{(k)}_j},\quad k\in\{0,1\}.$$

4) **Для каждого документа сэмплируйте длину и мешок слов**
- выберите длину $N_i$, например
$$N_i\sim \mathrm{Poisson}(80)+20;$$
- сэмплируйте счётчики:
$$x_i\mid (y_i=k)\sim \mathrm{Multinomial}(N_i,\theta_k).$$

5) **Верните $X,y$.**


In [88]:
from scipy.sparse import csr_matrix
def make_world_A(*, n=600, V=800, random_state=0):
    # TODO: реализуйте генератор мира A.
    # Подсказка: сделайте два распределения theta0, theta1 и сэмплируйте x_i ~ Multinomial(N_i, theta_{y_i}).
    np.random.seed(random_state)
    # генерация таргетов y ~ Bernoulli(0.5)
    y = np.random.binomial(1, 0.5, n)
    # индексы спама
    n_spam_trig = 40
    spam_inds = np.random.choice(np.arange(V), n_spam_trig, replace=False)
    # Веса для каждого класса
    w_0 = np.random.uniform(0.1, 1.0, size=V)
    w_1 = np.random.uniform(0.1, 1.0, size=V)
    w_1[spam_inds] *= 1.5 # веса для спама масштабируем
    # print(w_1[spam_inds])

    theta_0 = w_0 / w_0.sum()
    theta_1 = w_1 / w_1.sum()
    # print(theta_0.shape)
    # print(theta_1.shape)

    doc_len = np.random.poisson(80, n) + 20
    X = np.zeros(shape=(n, V), dtype=np.int64)
    for i in range(n):
      theta = theta_0 if y[i] == 0 else theta_1
      X[i, :] = np.random.multinomial(doc_len[i], theta)
    return X, y


In [89]:
make_world_A()

(array([[1, 0, 0, ..., 0, 0, 0],
        [0, 0, 1, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 1, 0],
        ...,
        [0, 0, 0, ..., 0, 1, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 array([1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1,
        0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0,
        1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
        1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1,
        0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0,
        0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1,
        0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0,
        0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1,
        1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 

In [90]:
# TODO: после реализации раскомментируйте
X_A, y_A, rep_A = run_world(make_world_A, "World A", n=600, V=800, seed=0)


=== World A ===
X shape: (600, 800) | class balance y.mean(): 0.49833333333333335
doc length: mean 99.55 std 8.966837049186667 | min 72 max 124
sparsity (fraction of zeros): 88.45%


,model,acc_mean,acc_std,logloss_mean,logloss_std
0,NB alpha=1,1.000000,0.000000,0.000002,0.000002
1,NB alpha=10,1.000000,0.000000,0.000024,0.000020
2,LogReg (scaling),1.000000,0.000000,0.007557,0.002922
3,NB alpha=1e-10,0.998333,0.003333,0.011761,0.023519


Наивный Байес показал себя лучшим по показателю кросс-энтропии. Это объясняется тем, что NB - это генеративная модель, которая делает предположение о том как устроены данные изнутри и знает каково истинное распределение. А данные мы специально подобрали под NB.

Логистическая Регрессия - дискриминационная модель. Её вероятностные скоры могут отличаться от истинных вероятностей, за счет чего  у неё выше logloss, который она напрямую минимизирует.

## Мир B: «Два типа спама» (мир, где NB мисспецифицирован)

### Цель
Сконструировать мир, где у класса spam **нет единого** распределения $\theta_1$: внутри spam есть два подтипа с разными словарями.

### Рецепт генерации

1) **Сгенерируйте метки**
$$y_i\sim \mathrm{Bernoulli}(0.5),\quad i=1,\dots,n.$$

2) **Постройте $\theta_0$ для ham**  
Как в мире A (можно использовать $S_{\text{ham}}$).

3) **Постройте два подтипа spam: $\theta_{1a},\theta_{1b}$**  
- выберите два множества триггеров
$$S_a,\ S_b\subset\{1,\dots,V\},$$
например по 20–50 слов каждое (желательно с малым пересечением);  
- постройте веса $w^{(1a)},w^{(1b)}$ и усиливайте их на $S_a$ и $S_b$ соответственно;  
- нормируйте:
$$\theta_{1a}=\frac{w^{(1a)}}{\sum_j w^{(1a)}_j},\qquad \theta_{1b}=\frac{w^{(1b)}}{\sum_j w^{(1b)}_j}.$$

4) **Выберите подтип для каждого spam-документа**  
Для документов с $y_i=1$ сэмплируйте
$$z_i\sim \mathrm{Bernoulli}(q)$$
(например, $q=0.7$).  
Если $z_i=1$, используйте $\theta_{1a}$, иначе $\theta_{1b}$.

5) **Сэмплируйте длину и мешок слов**
- $N_i\sim \mathrm{Poisson}(80)+20$ (или аналогично);  
- если $y_i=0$: $x_i\sim \mathrm{Multinomial}(N_i,\theta_0)$;  
- если $y_i=1$: $x_i\sim \mathrm{Multinomial}(N_i,\theta_{1a})$ или $\theta_{1b}$ по $z_i$.

6) **Верните $X,y$.**


In [116]:
def make_world_B(*, n=600, V=800, random_state=0):
    np.random.seed(random_state)
    y = np.random.binomial(1, 0.5, n)
    w_0 = np.random.uniform(0.1, 1.0, size=V)
    theta_0 = w_0 / w_0.sum()

    n_spam_trig = 50

    spam_inds = np.random.choice(V, 2*n_spam_trig, replace=False)

    S_a = spam_inds[:n_spam_trig-5]
    S_b = spam_inds[n_spam_trig+5:]
    # веса для положительного класса

    w1_a = np.random.uniform(0.1, 1.0, V)
    w1_a[S_a] *= 50
    w1_b = np.random.uniform(0.1, 1.0, V)
    w1_b[S_b] *= 50
    theta_1_a = w1_a / w1_a.sum()
    theta_1_b = w1_b / w1_b.sum()

    q = 0.9
    spam_subtype = np.random.binomial(1, q, n) # 1 -> тип А, 0 -> тип Б
    doc_lengths = np.random.poisson(80, n) + 20
    X = np.zeros((n, V), dtype=np.int64)

    for i in range(n):
        N_i = doc_lengths[i]

        if y[i] == 0:
            theta_current = theta_0
        else:
            if spam_subtype[i] == 1:
                theta_current = theta_1_a
            else:
                theta_current = theta_1_b


        X[i, :] = np.random.multinomial(N_i, theta_current)


    return X, y


In [117]:
# TODO: после реализации раскомментируйте
X_B, y_B, rep_B = run_world(make_world_B, "World B", n=600, V=800, seed=0)
# make_world_B()

=== World B ===
X shape: (600, 800) | class balance y.mean(): 0.49833333333333335
doc length: mean 100.63833333333334 std 8.926600541203927 | min 81 max 131
sparsity (fraction of zeros): 90.46%


,model,acc_mean,acc_std,logloss_mean,logloss_std
0,LogReg (scaling),1.000000,0.000000,0.000934,0.000166
1,NB alpha=10,0.998333,0.003333,0.002923,0.004790
2,NB alpha=1,0.991667,0.009129,0.024880,0.032065
3,NB alpha=1e-10,0.985000,0.009718,0.150116,0.135036


В мире $B$ в лидерах уже логистическая регрессия. На втором месте NB с сильным сглаживанием. Для NB нарушено его предположеие о данных: положительный класс имеет в себе два разнораспределенных вида спама. За счет этого кроссэнтропия выше. Но гиперпараметр сглаживания спасает ситуацию и NB c $alpha=10$ показывает достаточно хороший показатель logloss.

Логистическая регрессия не строит предположений о распределении подклассов, а просто старается их разделить, за счёт чего и logloss ниже.

## Мир C: «Редкие триггеры и роль $\alpha$» (сглаживание становится критичным)

### Цель
Сгенерировать **тот же тип мира**, что и A (один $\theta_0$ и один $\theta_1$), но в режиме сильной разреженности, чтобы влияние сглаживания $\alpha$ было заметным:
$$\hat\theta_{kj}=\frac{n_{kj}+\alpha}{n_{k\cdot}+V\alpha}.$$

### Рецепт генерации

1) **Сгенерируйте метки**
$$y_i\sim \mathrm{Bernoulli}(0.5),\quad i=1,\dots,n.$$

2) **Сделайте словарь большим и документы короткими**
- выберите $V\ge 800$;  
- выберите короткие длины, например
$$N_i\sim \mathrm{Poisson}(30)+10.$$

3) **Сделайте триггеры редкими**
- выберите маленькое множество
$$S_{\text{spam}}\subset\{1,\dots,V\}$$
размера 10–20;  
- постройте $\theta_0,\theta_1$ как в мире A, но **не делайте триггеры слишком сильными**, чтобы в среднем они встречались редко (порядка 0–1 раз на документ).

4) **Сэмплируйте мешок слов**
$$x_i\mid (y_i=k)\sim \mathrm{Multinomial}(N_i,\theta_k).$$

5) **Верните $X,y$.**  
Далее сравните `NB alpha=1e-10`, `NB alpha=1`, `NB alpha=10` и добейтесь заметной разницы по `logloss`.


In [118]:
import numpy as np

def make_world_C(*, n=600, V=2000, random_state=0):
    """
    Мир C: «Редкие триггеры и роль альфа».
    Возвращает плотный numpy массив (n, V).
    """
    np.random.seed(random_state)

    y = np.random.binomial(1, 0.5, n)

    doc_lengths = np.random.poisson(30, n) + 10

    n_spam_trig = 15  # Мало триггеров (10-20)
    spam_inds = np.random.choice(V, size=n_spam_trig, replace=False)

    w_0 = np.ones(V) * 0.1
    w_1 = np.ones(V) * 0.1

    w_1[spam_inds] *= 30.0

    theta_0 = w_0 / w_0.sum()
    theta_1 = w_1 / w_1.sum()

    X = np.zeros((n, V), dtype=np.int64)

    for i in range(n):
        N_i = doc_lengths[i]
        theta_current = theta_1 if y[i] == 1 else theta_0

        X[i, :] = np.random.multinomial(N_i, theta_current)

    return X, y

In [120]:
# TODO: после реализации раскомментируйте
X_C, y_C, rep_C = run_world(make_world_C, "World C", n=600, V=2000, seed=0)


=== World C ===
X shape: (600, 2000) | class balance y.mean(): 0.49833333333333335
doc length: mean 40.185 std 5.73853422051311 | min 25 max 61
sparsity (fraction of zeros): 98.05%


,model,acc_mean,acc_std,logloss_mean,logloss_std
0,NB alpha=1,0.941667,0.021731,0.175940,0.054017
1,NB alpha=10,0.926667,0.021344,0.198888,0.022081
2,LogReg (scaling),0.830000,0.023921,0.376685,0.067532
3,NB alpha=1e-10,0.745000,0.034400,4.368679,0.722163


В случае с большим объемом словаря и маленькими длинами документов параметр сглаживания играет важнейшую роль. Некоторые слова могут не присутствовать в каком-то классе, из-за чего числитель в формуле будет зануляться. Чтобы это не происходило, вводится сглаживание.

Хуже всего себя показал Байес с очень маленьким коээффициентом сглаживания. Лучше всего - с коэффициентом 1.

## Короткий отчёт (написать текстом)

Для каждого мира A/B/C добавьте 3–6 предложений:

- Какие предпосылки MultinomialNB выполняются/нарушаются?
- Почему это влияет на **logloss** (а не только на accuracy)?
- В мире C: почему alpha влияет? (сошлитесь на формулу $\hat\theta_{kj}=\frac{n_{kj}+\alpha}{n_{k\cdot}+V\alpha}$)
